# Day three — Species distribution modeling

---

-> summary: we have records of where species have been observed, but those points are not yet an explanation. Where is the species found? Where is it absent, or simply not recorded? What environmental conditions might explain the pattern? Once we have an idea of what determines the presence, absence, or even abundance of a species, we can go beyond observation, and model the species’ distribution. With such models, we can predict where we have no data, identify key regions for conservation, and even model what could happen if the environment were to change.

- PART I: From observations to ecological questions.
- PART II: Identifying environmental predictors: climate, land use, and habitat suitability.
- PART III: Species distribution modeling.
- PART IV: How to be a good scientist: tips, tricks, and pitfalls.


## *PART I: From observations to ecological questions*

## 📋 Planning note (instructor-facing — delete before making the student `day3.ipynb`)

**Target species: Philippine Flying Fox (*Pteropus vampyrus*).** 229 GBIF records (`gbif_flying_fox.csv`,
already cleaned in Day 2), spanning the *whole archipelago* (lon 118.5–126.1, lat 5.1–19.3) — genuinely
national in scope, unlike the Philippine Eagle (667 records, but **zero** fall within the Negros bbox, and
it's a range-restricted forest specialist — poorly suited to a first climate-driven model). The broader
`gbif_mammals_philippines.csv` (289 species) is a possible second/optional species later. This choice
matches the "Day 3 SDM target" note already left in `day2_bts.ipynb` for the flying fox pull.

**Data inventory & feasibility:**

| Data | Resolution | Extent | Status |
|---|---|---|---|
| Flying fox occurrences | point | Philippines-wide | ✅ have (`gbif_flying_fox.csv`) |
| Köppen-Geiger classification | 1 km (~0.0083°) | Philippines-wide | ✅ have (`kg_philippines_present.nc` + `..._future_ssp460.nc`) |
| Monthly precip/temp climatology | 0.1° | Philippines-wide | ✅ have (`climate_philippines_{1901_1930,1991_2020,2071_2099_ssp460}_mean.nc`) |
| Land-sea mask | 0.25° / 0.1° / 1 km | Philippines-wide | ✅ have (3 resolutions already) |
| Forest cover (Hansen) | 30 m | **Negros only** | ⚠️ partial — re-running Day 2's Hansen download for a few more 10°×10° tiles would cover the whole archipelago |
| Satellite imagery (Landsat/S2 composite) | ~100–200 m | **Negros only** | ⚠️ partial, same limitation |
| Elevation / terrain | — | — | ❌ not downloaded yet — feasible via the same Planetary Computer STAC pattern as Day 2's Landsat pull (`cop-dem-glo30` collection, public, no credentials) |
| Distance to coast | — | — | 🟡 free to derive from the land-sea mask we already have — no new download needed |

**Resolution strategy:** the layers that actually cover the flying fox's full range are the three
Philippines-wide ones (KG @ 1 km, monthly climate @ 0.1°, land-sea mask). The fine-resolution Negros-only
layers (Hansen 30 m, satellite ~100 m) don't extend past one island. A range-wide model has to work at the
**coarsest resolution among the layers it actually uses** (here, ~0.1°, ~11 km) — it can't invent detail
that isn't there. Concretely:
- **Core national-scale model (Parts II–III): climate-only predictors**, from the 0.1°/1 km layers.
- **Negros-specific forest/imagery data becomes an optional regional refinement / stretch goal** — a
  Negros-only overlay of yesterday's homemade land-cover classifier + Hansen forest-loss on top of the
  national suitability map. Discussed, likely cut for time.

**Scope note:** Day 3 should end up *shorter* than Day 2 (and some Day 2 content may migrate here later),
so Parts II–III stay lean — reuse already-processed data as directly as possible, and keep the model itself
simple (logistic regression on a handful of climate predictors; not a dedicated SDM package). This is also
the first time students see a real fitted statistical model rather than a hand-coded rule tree (Day 1/2's
"homemade classifiers") — a deliberate step up, not a break in style. Part IV should stay short discussion,
not new heavy code.

In [ ]:
# 1
import os
if not os.path.exists("/content/USLS-Workshop"):
    !git clone "https://github.com/Cas-Dec/USLS-Workshop.git" /content/USLS-Workshop
%cd /content/USLS-Workshop
!pip install -q .

from google.colab import drive
drive.mount('/content/drive')

**Plan.** Goal: get students to look critically at presence-only occurrence data *before* jumping to
modeling — nothing here needs new data, it's 100% reuse of Day 2's `gbif_flying_fox.csv` and mapping
patterns.

1. Reload `gbif_flying_fox.csv` (already cleaned/deduplicated in Day 2 — students already know this file).
2. Plot presence points over a Philippines outline (reuse the land-sea-mask contour pattern from Day 1/2
   maps). Prompt: does the species look randomly scattered, or clustered?
3. Discussion, pushing Day 2's sampling-bias theme specifically toward what it means *for modeling*:
   - GBIF gives us presences, not absences — an unrecorded area might be truly unsuitable, or just
     unsurveyed (near no roads/observers). We can't tell the difference from the points alone.
   - Introduce **background points / pseudo-absences** conceptually here (needed before Part III can fit
     anything): without real absences, we approximate them by sampling random points across the study area.
4. Land on the specific question Parts II-III will answer: *"Which combination of temperature and
   precipitation is associated with where flying foxes have been recorded, and can that predict suitability
   elsewhere in the country?"*

## *PART II: Identifying environmental predictors: climate, land use, and habitat suitability*

**Plan.** Goal: end up with one clean table — one row per point (presence + background), one column per
environmental variable.

1. Load `climate_philippines_1991_2020_mean.nc` (0.1°, 12-month precip + air_temperature climatology) and
   derive a small set of simple "bioclim-style" annual summaries per grid cell: annual mean temperature,
   annual total precipitation, temperature seasonality (e.g. max-min across the 12 months), precipitation
   seasonality. Technique-wise this is a direct callback to Day 1 (looping/aggregating over 12 monthly
   layers to derive KG classification rules) — no new xarray concept, just a new *use* of one.
2. Optionally bring in `kg_philippines_present.nc` (1 km) as a categorical predictor (climate zone).
3. Generate background/pseudo-absence points: random lon/lat draws over land only (reject using the
   land-sea mask), roughly matching the ~229 presences in count.
4. Extract each predictor's value at every presence + background point — nearest-grid-cell lookup, direct
   reuse of the `np.argmin(np.abs(array - value))` closest-index trick from Day 2's KG work.
5. Assemble into one `pandas.DataFrame`: `lon, lat, presence (0/1), temp_annual_mean, precip_annual_total,
   temp_seasonality, ...`.

**Optional stretch** (flagged "more data we'd need," likely cut for time):
- Extend Day 2's Hansen download to a few more 10°×10° tiles for Philippines-wide `treecover2000` @ 1 km
  as an extra predictor.
- Pull elevation from Copernicus DEM (`cop-dem-glo30`, Microsoft Planetary Computer — same STAC access
  pattern as Day 2's Landsat pull, no new mechanic to teach) at ~1 km.

## *PART III: Species distribution modeling*

**Plan.** Goal: fit and visualize a simple, homemade presence/background model — the same "rule → function
→ grid → map" pattern from Day 1 (climate classifier) and Day 2 (land-cover classifier), now applied to a
genuinely *predictive* (not just descriptive) use case.

1. Split Part II's table into predictors (`X`) and presence label (`y`).
2. Fit `sklearn.linear_model.LogisticRegression` (already in `environment.yml`, unused so far) on `X`/`y` —
   the simplest workable presence/background SDM. Deliberately not a dedicated SDM package, consistent with
   the workshop's homemade-first philosophy; this is the first time students fit a real statistical model
   rather than hand-coding a decision rule.
3. Predict suitability (predicted probability) across every land pixel of the 0.1° climate grid → a
   Philippines-wide habitat suitability map.
4. Plot it with the real presence points overlaid — sanity-check by eye: does the high-suitability zone
   actually contain most of the real records?
5. **Capstone, using data we already have:** re-predict the same fitted model onto
   `climate_philippines_2071_2099_ssp460_mean.nc` (future climate, already processed in Day 1) → a future
   suitability map. Directly ties Day 1's climate-change projection to a biodiversity consequence — "what
   does the model say happens to flying-fox habitat under future climate?" No new data required.
6. **Optional stretch:** zoom the same suitability map into the Negros bbox and overlay yesterday's
   land-cover classifier / Hansen forest-cover output — does finer-resolution data change the local picture?
   Time-permitting only.

## *PART IV: How to be a good scientist: tips, tricks, and pitfalls*

**Plan.** Short, discussion-heavy wrap-up (1-2 small demo cells at most) — new pitfalls specific to SDMs,
distinct from Day 2 Part IV's already-covered lessons (season/cloud confounding, single-snapshot vs.
multi-year products).

- **Presence-only bias**: GBIF records cluster near roads, cities, and observers — not necessarily near
  flying foxes. A model trained on biased presences learns the bias too.
- **Pseudo-absence sensitivity**: briefly show how the fitted suitability map shifts if background points
  are resampled differently (e.g. restricted to a smaller region vs. the whole country) — cheap and
  concrete.
- **Extrapolation risk**: when predicting onto 2071-2099 climate, check whether any future grid cells fall
  *outside* the range of climate values the model ever saw during training — real and checkable, since
  we're already doing exactly this projection in Part III.
- **Resolution/extent mismatches** (the same issue flagged in this notebook's opening planning note): a
  model built at 0.1° can't say anything about within-1km patchiness — tie back to Day 2's Hansen/Sentinel-2
  Negros data as a concrete example of the finer reality the coarse model can't see.
- **Correlation vs. causation, again** (echoing Day 1): climate correlates with suitability but doesn't
  capture roost-site specifics, food availability, hunting pressure, or dispersal limits.
- Close with the same "validate against an independent source" habit taught all workshop: compare the
  model's high-suitability zones against the species' known range informally.